In [1]:
# --------------------------
# author: Brian Kyanjo
# date: 2026-02-09
# description: This is an ICESEE-GHUB GUI notebook for rununing
#              ICESEE on cloud and cluster environments. It is 
#              designed to be run in a Jupyter Book environment.
# --------------------------

from IPython.display import HTML
HTML("""
<style>
/* Hide code inputs in Jupyter Book pages */
div.cell_input {display:none;}
</style>
""")

## ICESEE GUI

In [ ]:
from __future__ import annotations

import os, sys, subprocess, shutil
from pathlib import Path
from datetime import datetime
import yaml, re, time

import ipywidgets as W
from IPython.display import display, Image

# optional dependency for cluster mode
try:
    try:
        import paramiko
    except Exception:
        # install via pip
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "paramiko"])
        import paramiko
except Exception:
    paramiko = None

# ============================================================
# 0) Paths
# ============================================================
def find_repo_root():
    p = Path.cwd().resolve()
    while p != p.parent:
        if (p / "external" / "ICESEE").exists() and (p / "icesee_jupyter_book").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Could not locate repo root containing external/ICESEE and icesee_jupyter_book.")

REPO = find_repo_root()
BOOK = REPO / "icesee_jupyter_book"
EXT  = REPO / "external" / "ICESEE"

# ============================================================
# 1) Example registry (register examples here to show in dropdown)
# ============================================================
EXAMPLES = {
    "Lorenz-96 (fully runnable in GHUB)": dict(
        enabled=True,
        base=EXT / "applications" / "lorenz_model" / "examples" / "lorenz96",
        run_script="run_da_lorenz96.py",
        params="params.yaml",
        report_nb="read_results.ipynb",
        assets=["_modelrun_datasets"],   # copy into run dir for report, if needed
        model_name="lorenz",             # used for expected h5 name
        figures_dir="figures",           # where read_results typically saves images
    ),
    "ISSM (under development)":     dict(
        enabled=False,
        base=EXT / "applications" / "issm_model" / "examples" / "ISMIP_Choi",
        run_script="run_da_issm.py",
        params="params.yaml",
        report_nb="read_results.m",
        assets=["_modelrun_datasets"],
        model_name="issm",
        figures_dir="figures",
    ),
    "Flowline (under development)": dict(
        enabled=False, 
        base=EXT / "applications" / "flowline_model" / "examples" / "flowline_1d",
        run_script="run_da_flowline.py",
        params="params.yaml",
        report_nb="read_results.ipynb",
        assets=["_modelrun_datasets"],
        model_name="flowline",
        figures_dir="figures",
    ),
    "Icepack (under development)":  dict(
        enabled=False, 
        base=EXT / "applications" / "icepack_model" / "examples" / "synthetic_ice_stream",
        run_script="run_da_icepack.py",
        params="params.yaml",
        report_nb="read_results.ipynb",
        assets=["_modelrun_datasets"],
        model_name="icepack",
        figures_dir="figures",
    ),
    
}

def enabled_names():
    return [k for k,v in EXAMPLES.items() if v.get("enabled", False)]

# ============================================================
# 2) YAML helpers
# ============================================================
def load_yaml(p: Path) -> dict:
    return yaml.safe_load(p.read_text(encoding="utf-8")) or {}

def dump_yaml(d: dict, p: Path) -> None:
    p.write_text(yaml.safe_dump(d, sort_keys=False), encoding="utf-8")

def deep_update(d, u):
    for k, v in (u or {}).items():
        if isinstance(v, dict) and isinstance(d.get(k), dict):
            deep_update(d[k], v)
        else:
            d[k] = v
    return d

# ============================================================
# 3) Find runner + template
# ============================================================
def find_run_script(cfg: dict) -> Path:
    base = cfg["base"]
    rs = cfg.get("run_script")
    if rs and (base / rs).exists():
        return base / rs
    cands = list(base.rglob("run_da_*.py")) + list(base.rglob("run_*.py"))
    cands = [c for c in cands if c.is_file()]
    if not cands:
        raise FileNotFoundError(f"No run script found under {base}")
    cands.sort(key=lambda x: len(str(x)))
    return cands[0]

def find_params_template(cfg: dict) -> Path:
    wrapper_template = BOOK / "params.yaml"
    if wrapper_template.exists():
        return wrapper_template
    base = cfg["base"]
    p = base / (cfg.get("params") or "params.yaml")
    if p.exists():
        return p
    cands = list(base.rglob("params.yaml"))
    if not cands:
        raise FileNotFoundError(f"No params.yaml found under {base}")
    cands.sort(key=lambda x: len(str(x)))
    return cands[0]

def find_report_notebook(cfg: dict) -> Path | None:
    nb = cfg.get("report_nb")
    if not nb:
        return None
    p = cfg["base"] / nb
    return p if p.exists() else None

# ============================================================
# 4) Widget factory from YAML
# ============================================================
def widget_for(key: str, val):
    if isinstance(val, str):
        if key.lower() == "filter_type":
            opts = ["EnKF", "DEnKF", "EnTKF", "EnRSKF"]
            return W.Dropdown(options=opts, value=val if val in opts else opts[0], layout=W.Layout(width="100%"))
        if key.lower() in {"parallel_flag", "parallel"}:
            opts = ["serial", "MPI", "MPI_model"]
            return W.Dropdown(options=opts, value=val if val in opts else opts[0], layout=W.Layout(width="100%"))
        return W.Text(value=val, layout=W.Layout(width="100%"))

    if isinstance(val, bool):
        return W.Checkbox(value=val)

    if isinstance(val, int) and not isinstance(val, bool):
        return W.IntText(value=val, layout=W.Layout(width="100%"))
    if isinstance(val, float):
        return W.FloatText(value=val, layout=W.Layout(width="100%"))

    if isinstance(val, (list, dict)):
        return W.Textarea(
            value=yaml.safe_dump(val, sort_keys=False).strip(),
            layout=W.Layout(width="100%", height="110px"),
        )

    return W.Text(value=str(val), layout=W.Layout(width="100%"))

def read_widget(w):
    if isinstance(w, W.Textarea):
        try:
            return yaml.safe_load(w.value)
        except Exception:
            return w.value
    if hasattr(w, "value"):
        return w.value
    return None

# ============================================================
# 5) UI state
# ============================================================
example_dd = W.Dropdown(options=enabled_names(), value=enabled_names()[0], layout=W.Layout(width="320px"))

preset_dd  = W.Dropdown(options=["Default"], value="Default", layout=W.Layout(width="320px"))

# Filter algorithm (goes into params.yaml -> enkf-parameters/filter_type)
filter_alg_dd = W.Dropdown(
    options=[("EnKF", "EnKF"), ("DEnKF", "DEnKF"), ("EnTKF", "EnTKF"), ("EnRSKF", "EnRSKF")],
    value="EnKF",
    layout=W.Layout(width="320px")
)

# Output label controls expected result filename: results/<output>-<model>.h5
output_label_dd = W.Dropdown(
    options=[("true-wrong (demo output)", "true-wrong"), ("EnKF (output name)", "enkf")],
    value="true-wrong",
    layout=W.Layout(width="320px"),
)

ens_sl    = W.IntSlider(min=1, max=200, value=30, layout=W.Layout(width="320px"), continuous_update=False)
seed_in   = W.IntText(value=1, layout=W.Layout(width="320px"))

gen_report = W.Checkbox(value=True, description="Generate report (read_results.ipynb)")
open_latest= W.Checkbox(value=False, description="After run: open latest run folder")

run_btn   = W.Button(description="Run", button_style="success", icon="play")
clear_btn = W.Button(description="Clear", button_style="", icon="trash")

status_chip = W.HTML("<span class='icesee-status icesee-idle'>Idle</span>")

log_out = W.Output(layout=W.Layout(border="1px solid rgba(0,0,0,.12)", padding="10px", height="220px", overflow="auto"))
results_out = W.Output(layout=W.Layout(border="1px solid rgba(0,0,0,.12)", padding="10px", height="260px", overflow="auto"))

# params accordion (auto-built)
params_accordion = None
PARAMS0 = {}
WIDGETS = {}
EXTRA_YAML = {}
RUN_SCRIPT = None
TEMPLATE = None
REPORT_NB = None

# ssh client for cluster mode (optional) -----
run_mode = W.ToggleButtons(
    options=[("Local (GHUB)", "local"), ("Cluster (SSH + Slurm)", "cluster")],
    value="local",
    layout=W.Layout(width="420px"),
)

# ---- Cluster widgets ----
cluster_host = W.Text(value="", placeholder="login.cluster.edu", layout=W.Layout(width="320px"))
cluster_user = W.Text(value=os.environ.get("USER",""), placeholder="username", layout=W.Layout(width="320px"))
cluster_port = W.IntText(value=22, layout=W.Layout(width="120px"))

auth_mode = W.Dropdown(options=[("SSH Agent", "agent"), ("Key file", "keyfile")], value="agent", layout=W.Layout(width="180px"))
ssh_key_path = W.Text(value="~/.ssh/id_ed25519", layout=W.Layout(width="320px"))

remote_base_dir = W.Text(value="~/icesee-runs", layout=W.Layout(width="320px"))
remote_tag = W.Text(value="icesee", layout=W.Layout(width="220px"))

connect_btn = W.Button(description="Test SSH", icon="link", button_style="")
submit_btn  = W.Button(description="Submit to Slurm", icon="cloud-upload", button_style="warning")
status_btn  = W.Button(description="Check status", icon="search", button_style="")
tail_btn    = W.Button(description="Tail log", icon="file-text", button_style="")

# ---- Slurm widgets ----
slurm_job_name = W.Text(value="ICESEE", layout=W.Layout(width="220px"))
slurm_time     = W.Text(value="50:00:00", layout=W.Layout(width="220px"))
slurm_nodes    = W.IntText(value=1, layout=W.Layout(width="120px"))
slurm_ntasks   = W.IntText(value=40, layout=W.Layout(width="120px"))
slurm_tpn      = W.IntText(value=24, layout=W.Layout(width="120px"))
slurm_part     = W.Text(value="cpu-large", layout=W.Layout(width="220px"))
slurm_mem      = W.Text(value="256G", layout=W.Layout(width="220px"))
slurm_account  = W.Text(value="", placeholder="e.g. gts-arobel3-atlas", layout=W.Layout(width="220px"))
slurm_mail     = W.Text(value="", placeholder="your@email", layout=W.Layout(width="220px"))

# Runtime knobs that map to your command line
cluster_mpi_np      = W.IntText(value=40, layout=W.Layout(width="120px"))
cluster_model_nprocs= W.IntText(value=4, layout=W.Layout(width="120px"))

# Keep state for last submission
LAST_REMOTE_RUN = {"dir": None, "jobid": None}

# --
# ============================================================
# Run mode (Local vs Cluster)
# ============================================================
run_mode = W.ToggleButtons(
    options=[("Local (GHUB)", "local"), ("Cluster (SSH + Slurm)", "cluster")],
    value="local",
    button_style="",
    tooltips=["Run directly in this notebook", "Submit a Slurm job via SSH"],
    layout=W.Layout(width="320px")
)

# Placeholder cluster panel/card (create this later with real fields)
cluster_panel = W.VBox([
    W.HTML("<div class='icesee-h'>Cluster settings</div>"),
    W.HTML("<div class='icesee-subtle'>SSH + Slurm options will appear here.</div>")
])
cluster_panel.add_class("icesee-card")

def _toggle_cluster_panel(_=None):
    cluster_panel.layout.display = "block" if run_mode.value == "cluster" else "none"

run_mode.observe(_toggle_cluster_panel, names="value")
_toggle_cluster_panel()
# ---end of cluster widgets---

# ============================================================
# 6) Build params UI from template
# ============================================================
def build_params_ui(template_path: Path):
    global params_accordion, PARAMS0, WIDGETS, EXTRA_YAML
    PARAMS0 = load_yaml(template_path)
    WIDGETS = {}
    EXTRA_YAML = {}

    children, titles = [], []

    for sec, sec_dict in (PARAMS0 or {}).items():
        titles.append(sec)
        sec_widgets = {}
        rows = []

        if isinstance(sec_dict, dict):
            for k, v in sec_dict.items():
                w = widget_for(k, v)
                sec_widgets[k] = w
                rows.append(W.HBox([
                    W.HTML(f"<div class='icesee-k'>{k}</div>"),
                    w
                ], layout=W.Layout(gap="12px")))

            extra = W.Textarea(
                value="# Add future keys here (YAML)\n",
                layout=W.Layout(width="100%", height="90px")
            )
            EXTRA_YAML[sec] = extra
            rows.append(W.HTML("<div class='icesee-subtle' style='margin-top:6px'>Extra keys (optional)</div>"))
            rows.append(extra)
        else:
            w = W.Textarea(value=yaml.safe_dump(sec_dict, sort_keys=False).strip(),
                           layout=W.Layout(width="100%", height="140px"))
            sec_widgets["__raw__"] = w
            rows.append(w)

        WIDGETS[sec] = sec_widgets
        children.append(W.VBox(rows, layout=W.Layout(gap="8px")))

    params_accordion = W.Accordion(children=children)
    for i, t in enumerate(titles):
        params_accordion.set_title(i, t)

def sync_quick_into_widgets():
    # Try section naming variants
    sec = None
    for candidate in ["enkf-parameters", "enkf_parameters", "enkf"]:
        if candidate in WIDGETS:
            sec = candidate
            break
    if not sec:
        return

    if "Nens" in WIDGETS[sec]:
        WIDGETS[sec]["Nens"].value = int(ens_sl.value)
    if "seed" in WIDGETS[sec]:
        WIDGETS[sec]["seed"].value = int(seed_in.value)
    if "filter_type" in WIDGETS[sec]:
        WIDGETS[sec]["filter_type"].value = str(filter_alg_dd.value)

def build_config_from_widgets() -> dict:
    cfg = {}
    for sec, sw in WIDGETS.items():
        if "__raw__" in sw:
            cfg[sec] = yaml.safe_load(sw["__raw__"].value)
            continue

        cfg[sec] = {}
        for k, w in sw.items():
            if k == "__raw__":
                continue
            cfg[sec][k] = read_widget(w)

        extra = EXTRA_YAML.get(sec)
        if extra:
            txt = extra.value.strip()
            if txt and not txt.startswith("#"):
                extra_obj = yaml.safe_load(txt) or {}
                if isinstance(extra_obj, dict):
                    cfg[sec].update(extra_obj)
                else:
                    cfg[sec]["__extra__"] = extra_obj
    return cfg

# ============================================================
# 7) Run helpers
# ============================================================
def run_dir() -> Path:
    rd = BOOK / "icesee_runs" / datetime.now().strftime("%Y%m%d_%H%M%S")
    rd.mkdir(parents=True, exist_ok=True)
    (rd / "results").mkdir(exist_ok=True)
    (rd / "figures").mkdir(exist_ok=True)
    return rd

def expected_h5_path(rd: Path, cfg: dict) -> Path:
    model_name = cfg.get("model_name", "lorenz")
    return rd / "results" / f"{output_label_dd.value}-{model_name}.h5"

def set_status(state: str):
    cls = {"idle":"icesee-idle","running":"icesee-running","done":"icesee-done","fail":"icesee-fail"}[state]
    label = {"idle":"Idle","running":"Running…","done":"Done","fail":"Failed"}[state]
    status_chip.value = f"<span class='icesee-status {cls}'>{label}</span>"

def force_external_icesee_env():
    external_dir = (REPO / "external").resolve()
    env = os.environ.copy()
    env["PYTHONPATH"] = f"{external_dir}{os.pathsep}{env.get('PYTHONPATH','')}"
    env["PYTHONNOUSERSITE"] = "1"
    return env, external_dir

def mirror_assets_for_report(example_cfg: dict, rd: Path):
    # Some report notebooks expect folders like _modelrun_datasets present.
    base = example_cfg["base"]
    for a in example_cfg.get("assets", []):
        src = base / a
        if src.exists():
            dst = rd / a
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)

def run_report_notebook(example_cfg: dict, rd: Path):
    nb = REPORT_NB
    if not nb or not nb.exists():
        with log_out:
            print("[wrapper] No report notebook configured/found; skipping read_results.")
        return None

    try:
        import papermill as pm
    except Exception as e:
        raise RuntimeError("papermill is required to run read_results.ipynb automatically. "
                           "Install it (pip install papermill) or ask me for an nbclient version.") from e

    nb_out = rd / "report.ipynb"
    mirror_assets_for_report(example_cfg, rd)

    # Execute report with cwd=rd so relative paths resolve:
    # - results/<output>-lorenz.h5
    # - figures/...
    pm.execute_notebook(
        input_path=str(nb),
        output_path=str(nb_out),
        cwd=str(rd),
        log_output=True,
    )
    return nb_out

def refresh_results_preview(rd: Path):
    results_out.clear_output()
    with results_out:
        # prefer real report figures
        fig_dir = rd / "figures"
        pngs = sorted(fig_dir.glob("*.png"))
        if not pngs:
            pngs = sorted((rd / "results").glob("*.png"))

        h5s  = sorted((rd / "results").glob("*.h5"))

        print("Run folder:", rd)
        print(f"Results: {len(h5s)} H5, {len(pngs)} PNG\n")

        if h5s:
            for p in h5s[:10]:
                print(" -", p.name)

        if pngs:
            print("\nFigures:")
            for p in pngs[:6]:
                display(Image(filename=str(p)))
        else:
            print("\nNo figures found yet.")

# ------ ssh client for cluster mode (optional dependency) ------
def sh_quote(s: str) -> str:
    return "'" + s.replace("'", "'\"'\"'") + "'"

class SlurmSSHClient:
    def __init__(self, host, username, port=22, auth_mode="agent", key_path=None, timeout=20):
        if paramiko is None:
            raise RuntimeError("paramiko is not installed. Install with: pip install paramiko")
        self.host = host
        self.username = username
        self.port = int(port)
        self.auth_mode = auth_mode
        self.key_path = key_path
        self.timeout = timeout
        self.client = None

    def connect(self):
        c = paramiko.SSHClient()
        c.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        if self.auth_mode == "agent":
            c.connect(self.host, port=self.port, username=self.username,
                      timeout=self.timeout, allow_agent=True, look_for_keys=True)
        elif self.auth_mode == "keyfile":
            if not self.key_path:
                raise ValueError("key_path required for auth_mode=keyfile")
            key_path = os.path.expanduser(self.key_path)
            if not os.path.exists(key_path):
                raise FileNotFoundError(f"SSH key not found: {key_path}")
            # try common key types
            pkey = None
            for KeyCls in (paramiko.Ed25519Key, paramiko.RSAKey, paramiko.ECDSAKey):
                try:
                    pkey = KeyCls.from_private_key_file(key_path)
                    break
                except Exception:
                    pass
            if pkey is None:
                raise ValueError("Could not load private key (unsupported/encrypted key).")
            c.connect(self.host, port=self.port, username=self.username, pkey=pkey,
                      timeout=self.timeout, allow_agent=False, look_for_keys=False)
        else:
            raise ValueError("auth_mode must be 'agent' or 'keyfile'")
        self.client = c
        return self

    def close(self):
        if self.client:
            self.client.close()
            self.client = None

    def run(self, cmd):
        if not self.client:
            raise RuntimeError("Not connected")
        stdin, stdout, stderr = self.client.exec_command(cmd)
        out = stdout.read().decode("utf-8", errors="replace")
        err = stderr.read().decode("utf-8", errors="replace")
        code = stdout.channel.recv_exit_status()
        return code, out, err

    def mkdir(self, path):
        code, out, err = self.run(f"mkdir -p {sh_quote(path)}")
        if code != 0:
            raise RuntimeError(err)
        return True

    def upload_text(self, remote_path, text):
        if not self.client:
            raise RuntimeError("Not connected")
        sftp = self.client.open_sftp()
        try:
            with sftp.file(remote_path, "w") as f:
                f.write(text)
        finally:
            sftp.close()

    def upload_file(self, local_path, remote_path):
        if not self.client:
            raise RuntimeError("Not connected")
        sftp = self.client.open_sftp()
        try:
            sftp.put(str(local_path), remote_path)
        finally:
            sftp.close()

def make_remote_run_dir(base_dir="~/icesee-runs", tag="icesee"):
    ts = time.strftime("%Y%m%d-%H%M%S")
    return f"{base_dir.rstrip('/')}/{tag}-{ts}"

SLURM_TEMPLATE = """#!/bin/sh
#SBATCH -t{{TIME}}
#SBATCH -J{{JOB_NAME}}
#SBATCH -N {{NODES}}
#SBATCH -n {{NTASKS}}
#SBATCH --ntasks-per-node={{TPN}}
#SBATCH --partition={{PARTITION}}
#SBATCH --mem={{MEM}}
#SBATCH -A {{ACCOUNT}}
#SBATCH -o {{OUTFILE}}
#SBATCH --mail-type=BEGIN,END,FAIL
#SBATCH --mail-user={{MAIL_USER}}

cd $SLURM_SUBMIT_DIR
module purge
module load {{MATLAB_MODULE}}
module load {{GCC_MODULE}}

# User/project paths (edit if needed)
export ISSM_DIR={{ISSM_DIR}}

# OpenMPI external installed by ICESEE-Spack
export LD_LIBRARY_PATH={{OPENMPI_PREFIX}}/lib:$LD_LIBRARY_PATH
export PATH={{OPENMPI_PREFIX}}/bin:$PATH

# Spack env activation script (optional)
{{SPACK_ACTIVATE_LINE}}

# Run
mpirun -mca coll ^hcoll -np {{MPI_NP}} python3 {{RUN_SCRIPT}} -F params.yaml --Nens={{NENS}} --model_nprocs={{MODEL_NPROCS}}
"""

def render_slurm_script(d: dict) -> str:
    txt = SLURM_TEMPLATE
    for k, v in d.items():
        txt = txt.replace("{{"+k+"}}", str(v))
    return txt
# ---- end of SlurmSSHClient ----

# ============================================================
# 8) Run
# ============================================================
def run_example_local():
    example_cfg = EXAMPLES[example_dd.value]

    # push quick knobs into template widgets before writing params.yaml
    sync_quick_into_widgets()

    cfg = build_config_from_widgets()

    rd = run_dir()
    params_path = rd / "params.yaml"
    dump_yaml(cfg, params_path)

    env, external_dir = force_external_icesee_env()

    cmd = [sys.executable, str(RUN_SCRIPT), "-F", str(params_path)]

    set_status("running")
    log_out.clear_output()
    with log_out:
        print("Example :", example_dd.value)
        print("Runner  :", RUN_SCRIPT)
        print("Report  :", REPORT_NB if REPORT_NB else "(none)")
        print("CWD     :", rd)
        print("Command :", " ".join(cmd))
        print("PYTHONPATH(prepended):", external_dir)
        print("-"*70)

    proc = subprocess.Popen(
        cmd,
        cwd=str(rd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    assert proc.stdout is not None
    full_log = []
    for line in proc.stdout:
        full_log.append(line)
        with log_out:
            print(line.rstrip())
    rc = proc.wait()

    log_text = "".join(full_log)
    looks_like_failure = (
        "Traceback (most recent call last)" in log_text
        or "Error in serial run mode" in log_text
        or "UnboundLocalError" in log_text
        or "FileNotFoundError" in log_text
    )

    with log_out:
        print("-"*70)
        print("Return code:", rc)

    # even if rc=0, treat tracebacks as failure
    if rc != 0 or looks_like_failure:
        with log_out:
            if rc == 0 and looks_like_failure:
                print("[GUI] Detected error in output despite rc=0 -> marking as FAILED.")
        set_status("fail")
        refresh_results_preview(rd)
        return

    # Ensure expected H5 exists where report expects it.
    exp_h5 = expected_h5_path(rd, example_cfg)
    if not exp_h5.exists():
        # try to locate it anywhere under external/ICESEE and copy back
        model_name = example_cfg.get("model_name", "lorenz")
        h5_name = f"{output_label_dd.value}-{model_name}.h5"
        candidates = list(EXT.glob(f"**/results/{h5_name}"))
        candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
        if candidates:
            exp_h5.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(candidates[0], exp_h5)
            with log_out:
                print(f"[wrapper] Copied H5 into run folder: {exp_h5.name}")
        else:
            with log_out:
                print(f"[wrapper] WARNING: expected H5 not found: {exp_h5}")
            # still continue; maybe report uses a different filename

    # Run report notebook to generate the REAL figures
    if gen_report.value:
        try:
            with log_out:
                print("[wrapper] Running report notebook (read_results.ipynb)…")
            run_report_notebook(example_cfg, rd)
            with log_out:
                print("[wrapper] Report done.")
        except Exception as e:
            with log_out:
                print("[wrapper] Report failed:", type(e).__name__, e)
            set_status("fail")
            refresh_results_preview(rd)
            return
        
    set_status("done")
    refresh_results_preview(rd)

    with log_out:
        print("\nRun folder:", rd)

def run_example_cluster():
    example_cfg = EXAMPLES[example_dd.value]

    # For now, only enable cluster for ISSM once you flip enabled=True later.
    # You can still submit any run script, but this warns you.
    if "ISSM" not in example_dd.value:
        with log_out:
            print("[cluster] NOTE: cluster mode is mainly intended for ISSM. Submitting anyway...")

    if not cluster_host.value.strip() or not cluster_user.value.strip():
        with log_out:
            print("[cluster][ERROR] cluster_host and cluster_user are required.")
        set_status("fail")
        return

    # Push quick knobs into params
    sync_quick_into_widgets()
    cfg_yaml = build_config_from_widgets()

    # Local staging dir (we use the existing run_dir() for params.yaml)
    rd = run_dir()
    params_path = rd / "params.yaml"
    dump_yaml(cfg_yaml, params_path)

    # Remote run dir
    rdir = make_remote_run_dir(remote_base_dir.value.strip(), remote_tag.value.strip() or "icesee")

    # Build slurm script
    job_name = slurm_job_name.value.strip() or "ICESEE"
    outfile = f"steadystate-%j.out"

    # Optional spack activate line: user can keep empty, or we can set later.
    spack_activate_line = ""  # keep blank for baby-step

    slurm_vars = {
        "TIME": slurm_time.value.strip(),
        "JOB_NAME": job_name,
        "NODES": int(slurm_nodes.value),
        "NTASKS": int(slurm_ntasks.value),
        "TPN": int(slurm_tpn.value),
        "PARTITION": slurm_part.value.strip(),
        "MEM": slurm_mem.value.strip(),
        "ACCOUNT": slurm_account.value.strip() or "REPLACE_ME",
        "OUTFILE": outfile,
        "MAIL_USER": slurm_mail.value.strip() or "REPLACE_ME",
        "MATLAB_MODULE": "matlab/r2024b",
        "GCC_MODULE": f"gcc/{WANT_GCC}" if "WANT_GCC" in globals() else "gcc/13",
        "ISSM_DIR": "${ISSM_DIR}",  # user module/env sets it (we will wire later)
        "OPENMPI_PREFIX": "${OPENMPI_PREFIX}",
        "SPACK_ACTIVATE_LINE": spack_activate_line,
        "MPI_NP": int(cluster_mpi_np.value),
        "RUN_SCRIPT": Path(find_run_script(example_cfg)).name,
        "NENS": int(ens_sl.value),
        "MODEL_NPROCS": int(cluster_model_nprocs.value),
    }

    slurm_text = render_slurm_script(slurm_vars)

    set_status("running")
    log_out.clear_output()
    with log_out:
        print("[cluster] Example:", example_dd.value)
        print("[cluster] Remote dir:", rdir)

    # Connect and stage
    ssh = None
    try:
        ssh = SlurmSSHClient(
            host=cluster_host.value.strip(),
            username=cluster_user.value.strip(),
            port=int(cluster_port.value),
            auth_mode=auth_mode.value,
            key_path=ssh_key_path.value.strip() if auth_mode.value == "keyfile" else None,
        ).connect()

        ssh.mkdir(rdir)

        # Upload params.yaml
        ssh.upload_file(params_path, f"{rdir}/params.yaml")

        # Upload run script (we upload the actual python file from the example folder)
        run_script_path = find_run_script(example_cfg)
        ssh.upload_file(run_script_path, f"{rdir}/{run_script_path.name}")

        # Upload slurm script
        ssh.upload_text(f"{rdir}/slurm_run.sh", slurm_text)
        ssh.run(f"chmod +x {sh_quote(rdir+'/slurm_run.sh')}")

        # Submit
        code, out, err = ssh.run(f"cd {sh_quote(rdir)} && sbatch slurm_run.sh")
        with log_out:
            print("[cluster] sbatch:", out.strip())
            if err.strip():
                print("[cluster] STDERR:", err.strip())

        m = re.search(r"Submitted batch job\\s+(\\d+)", out)
        if not m:
            raise RuntimeError(f"Could not parse JobID from sbatch output: {out} {err}")
        jobid = m.group(1)

        LAST_REMOTE_RUN["dir"] = rdir
        LAST_REMOTE_RUN["jobid"] = jobid

        with log_out:
            print("[cluster] JobID:", jobid)
            print("[cluster] Next: use 'Check status' or 'Tail log'.")

        set_status("done")

    except Exception as e:
        set_status("fail")
        with log_out:
            print("[cluster][ERROR]", type(e).__name__, e)
    finally:
        if ssh:
            ssh.close()

def run_example():
    if run_mode.value == "cluster":
        return run_example_cluster()
    return run_example_local()

# ============================================================
# 9) Rebuild on example change
# ============================================================
def rebuild_for_example(_=None):
    global RUN_SCRIPT, TEMPLATE, REPORT_NB

    cfg = EXAMPLES[example_dd.value]
    RUN_SCRIPT = find_run_script(cfg)
    TEMPLATE   = find_params_template(cfg)
    REPORT_NB  = find_report_notebook(cfg)

    build_params_ui(TEMPLATE)
    params_holder.children = (params_accordion,)

    with log_out:
        print("[Loaded]")
        print("Template:", TEMPLATE)
        print("Runner  :", RUN_SCRIPT)
        print("Report  :", REPORT_NB if REPORT_NB else "(none)")

example_dd.observe(rebuild_for_example, names="value")
run_btn.on_click(lambda b: run_example())
clear_btn.on_click(lambda b: (log_out.clear_output(), results_out.clear_output(), set_status("idle")))

def on_test_ssh(_):
    log_out.clear_output()
    if not cluster_host.value.strip() or not cluster_user.value.strip():
        with log_out: print("[cluster] Provide host + user first.")
        return
    ssh = None
    try:
        ssh = SlurmSSHClient(
            host=cluster_host.value.strip(),
            username=cluster_user.value.strip(),
            port=int(cluster_port.value),
            auth_mode=auth_mode.value,
            key_path=ssh_key_path.value.strip() if auth_mode.value == "keyfile" else None,
        ).connect()
        rdir = make_remote_run_dir(remote_base_dir.value.strip(), "test")
        ssh.mkdir(rdir)
        code, out, err = ssh.run(f"cd {sh_quote(rdir)} && hostname && pwd")
        with log_out:
            print("[cluster] OK")
            print(out.strip())
    except Exception as e:
        with log_out:
            print("[cluster][ERROR]", type(e).__name__, e)
    finally:
        if ssh: ssh.close()

def on_check_status(_):
    rdir = LAST_REMOTE_RUN.get("dir")
    jobid = LAST_REMOTE_RUN.get("jobid")
    if not jobid:
        with log_out: print("[cluster] No JobID yet. Submit first.")
        return
    ssh = None
    try:
        ssh = SlurmSSHClient(cluster_host.value.strip(), cluster_user.value.strip(), int(cluster_port.value),
                             auth_mode=auth_mode.value,
                             key_path=ssh_key_path.value.strip() if auth_mode.value == "keyfile" else None).connect()
        code, out, err = ssh.run(f"squeue -j {jobid} -o '%i %T %M %D %R'")
        with log_out:
            print("[cluster] squeue:")
            print(out.strip() or "(not in queue)")
            if err.strip():
                print("STDERR:", err.strip())
    except Exception as e:
        with log_out:
            print("[cluster][ERROR]", type(e).__name__, e)
    finally:
        if ssh: ssh.close()

def on_tail_log(_):
    rdir = LAST_REMOTE_RUN.get("dir")
    jobid = LAST_REMOTE_RUN.get("jobid")
    if not rdir or not jobid:
        with log_out: print("[cluster] No remote run dir / JobID. Submit first.")
        return
    ssh = None
    try:
        ssh = SlurmSSHClient(cluster_host.value.strip(), cluster_user.value.strip(), int(cluster_port.value),
                             auth_mode=auth_mode.value,
                             key_path=ssh_key_path.value.strip() if auth_mode.value == "keyfile" else None).connect()
        # stdout file name matches template: steadystate-%j.out
        # after submission, file becomes steadystate-<jobid>.out
        out_file = f"{rdir}/steadystate-{jobid}.out"
        code, out, err = ssh.run(f"test -f {sh_quote(out_file)} && tail -n 80 {sh_quote(out_file)} || echo 'log not yet created'")
        with log_out:
            print("[cluster] tail:", out_file)
            print(out.rstrip())
            if err.strip():
                print("STDERR:", err.strip())
    except Exception as e:
        with log_out:
            print("[cluster][ERROR]", type(e).__name__, e)
    finally:
        if ssh: ssh.close()

connect_btn.on_click(on_test_ssh)
status_btn.on_click(on_check_status)
tail_btn.on_click(on_tail_log)
submit_btn.on_click(lambda b: run_example_cluster())


# ============================================================
# 10) Pro UX CSS
# ============================================================
css = """
<style>
.icesee-wrap { font-family: system-ui, -apple-system, Segoe UI, Roboto, Arial; }
.icesee-title { font-size: 18px; font-weight: 700; margin: 6px 0 4px; }
.icesee-subtitle { color: rgba(0,0,0,.65); margin-bottom: 14px; }
.icesee-card { border: 1px solid rgba(0,0,0,.10); border-radius: 12px; padding: 14px; background: #fff; }
.icesee-h { font-size: 18px; font-weight: 800; margin: 2px 0 10px; }
.icesee-lbl { width: 120px; font-weight: 650; }
.icesee-k { width: 220px; font-weight: 650; color: rgba(0,0,0,.78); }
.icesee-subtle { color: rgba(0,0,0,.60); font-size: 12px; }
.icesee-status { display:inline-block; padding: 8px 14px; border-radius: 999px; font-weight: 700; border: 1px solid rgba(0,0,0,.10); }
.icesee-idle { background: rgba(0,0,0,.04); }
.icesee-running { background: rgba(16, 122, 255, .12); }
.icesee-done { background: rgba(30, 170, 80, .14); }
.icesee-fail { background: rgba(220, 60, 60, .14); }
</style>
"""
display(W.HTML(css))

# ============================================================
# 11) Layout (your pro UX)
# ============================================================
header = W.HTML(
    "<div class='icesee-wrap'>"
    "<div class='icesee-title'>Run ICESEE examples with one click.</div>"
    "<div class='icesee-subtitle'>Outputs and reports are saved to the wrapper and shown on the right.</div>"
    "</div>"
)

left = W.VBox([
    W.HTML("<div class='icesee-h'>Run settings</div>"),
    W.HBox([W.HTML("<div class='icesee-lbl'>Mode:</div>"), run_mode], layout=W.Layout(gap="12px")),
    W.HBox([W.HTML("<div class='icesee-lbl'>Example:</div>"), example_dd], layout=W.Layout(gap="12px")),
    W.HBox([W.HTML("<div class='icesee-lbl'>Preset:</div>"),  preset_dd],  layout=W.Layout(gap="12px")),
    W.HBox([W.HTML("<div class='icesee-lbl'>Filter:</div>"),  filter_alg_dd],  layout=W.Layout(gap="12px")),
    W.HBox([W.HTML("<div class='icesee-lbl'>Output:</div>"),  output_label_dd], layout=W.Layout(gap="12px")),
    W.HBox([W.HTML("<div class='icesee-lbl'>Ens:</div>"),     ens_sl],     layout=W.Layout(gap="12px")),
    W.HBox([W.HTML("<div class='icesee-lbl'>Seed:</div>"),    seed_in],    layout=W.Layout(gap="12px")),
    W.Box([gen_report], layout=W.Layout(margin="6px 0 0 120px")),
    W.Box([open_latest], layout=W.Layout(margin="0 0 8px 120px")),
    W.HTML("<div class='icesee-subtle' style='margin:8px 0 8px'>Full configuration (from <code>params.yaml</code>)</div>"),
])
cluster_panel = W.VBox([
    W.HTML("<div class='icesee-h'>Cluster (SSH + Slurm)</div>"),
    W.HBox([W.HTML("<div class='icesee-lbl'>Run mode:</div>"), run_mode], layout=W.Layout(gap="12px")),

    W.HTML("<div class='icesee-subtle'>SSH connection</div>"),
    W.HBox([W.HTML("<div class='icesee-lbl'>Host:</div>"), cluster_host], layout=W.Layout(gap="12px")),
    W.HBox([W.HTML("<div class='icesee-lbl'>User:</div>"), cluster_user, W.HTML("<div class='icesee-lbl'>Port:</div>"), cluster_port], layout=W.Layout(gap="12px")),
    W.HBox([W.HTML("<div class='icesee-lbl'>Auth:</div>"), auth_mode, W.HTML("<div class='icesee-lbl'>Key:</div>"), ssh_key_path], layout=W.Layout(gap="12px")),
    W.HBox([W.HTML("<div class='icesee-lbl'>Remote dir:</div>"), remote_base_dir, W.HTML("<div class='icesee-lbl'>Tag:</div>"), remote_tag], layout=W.Layout(gap="12px")),
    W.HBox([connect_btn, submit_btn, status_btn, tail_btn], layout=W.Layout(gap="10px")),

    W.HTML("<div class='icesee-subtle' style='margin-top:8px'>Slurm resources</div>"),
    W.HBox([W.HTML("<div class='icesee-lbl'>Job:</div>"), slurm_job_name, W.HTML("<div class='icesee-lbl'>Time:</div>"), slurm_time], layout=W.Layout(gap="12px")),
    W.HBox([W.HTML("<div class='icesee-lbl'>Nodes:</div>"), slurm_nodes, W.HTML("<div class='icesee-lbl'>Tasks:</div>"), slurm_ntasks, W.HTML("<div class='icesee-lbl'>TPN:</div>"), slurm_tpn], layout=W.Layout(gap="12px")),
    W.HBox([W.HTML("<div class='icesee-lbl'>Part:</div>"), slurm_part, W.HTML("<div class='icesee-lbl'>Mem:</div>"), slurm_mem], layout=W.Layout(gap="12px")),
    W.HBox([W.HTML("<div class='icesee-lbl'>Acct:</div>"), slurm_account, W.HTML("<div class='icesee-lbl'>Mail:</div>"), slurm_mail], layout=W.Layout(gap="12px")),
    W.HBox([W.HTML("<div class='icesee-lbl'>MPI np:</div>"), cluster_mpi_np, W.HTML("<div class='icesee-lbl'>Model nprocs:</div>"), cluster_model_nprocs], layout=W.Layout(gap="12px")),
])
cluster_panel.add_class("icesee-card")
def _toggle_cluster_panel(_=None):
    cluster_panel.layout.display = "block" if run_mode.value == "cluster" else "none"

run_mode.observe(_toggle_cluster_panel, names="value")
_toggle_cluster_panel()   # set initial state
def _toggle_cluster_buttons(_=None):
    is_cluster = (run_mode.value == "cluster")
    submit_btn.disabled = not is_cluster
    status_btn.disabled = not is_cluster
    tail_btn.disabled   = not is_cluster
    connect_btn.disabled= not is_cluster

run_mode.observe(_toggle_cluster_buttons, names="value")
_toggle_cluster_buttons()

params_holder = W.VBox([])
# left_card = W.VBox([left, params_holder])
left_card = W.VBox([left, cluster_panel, params_holder], layout=W.Layout(gap="26px"))
left_card.add_class("icesee-card")

right = W.VBox([
    W.HTML("<div class='icesee-h'>Run log</div>"),
    log_out,
    W.HTML("<div class='icesee-h' style='margin-top:14px'>Results preview</div>"),
    results_out,
])
right_card = W.VBox([right])
right_card.add_class("icesee-card")

actions = W.HBox([run_btn, clear_btn, status_chip], layout=W.Layout(gap="12px"))
actions_card = W.VBox([W.HTML("<div class='icesee-h'>Status</div>"), actions])
actions_card.add_class("icesee-card")

page = W.VBox([
    header,
    W.HBox([left_card, right_card], layout=W.Layout(gap="26px")),
    actions_card
])


display(page)
set_status("idle")
rebuild_for_example()

# Keep template in sync with the quick knobs:
def _sync_knobs(_=None):
    # map filter_alg into enkf-parameters/filter_type etc.
    sync_quick_into_widgets()

filter_alg_dd.observe(_sync_knobs, names="value")
ens_sl.observe(_sync_knobs, names="value")
seed_in.observe(_sync_knobs, names="value")

HTML(value='\n<style>\n.icesee-wrap { font-family: system-ui, -apple-system, Segoe UI, Roboto, Arial; }\n.ices…